# 08. Streamlit & Packaging: Strategic Pipeline Deployment

This notebook consolidates the entire logic—from raw data ingestion to risk probability—into a single **scikit-learn Pipeline** artifact (`full_pipeline_14day_strategic.pkl`).

**🎯 THE MLOPS "BLACK-BOX"**
By encapsulating the `ColumnTransformer` (OneHot & Target Encoding) and the `XGBoost` champion model into one object, we ensure:
1. **Training-Serving Consistency**: Streamlit will use the exact same logic we used during training.
2. **Simplified Inference**: We can feed raw, unencoded business data directly into the pipeline.
3. **14-Day Strategic Alignment**: This artifact is calibrated with our re-engineered signal, moving away from reactive alerts to proactive international replenishment management.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# 1. LOAD RAW DATA
RAW_DATA_PATH = '/Users/rober/retail-stockout-risk-scoring/02_Data/01_Raw/retail_store_inventory.csv'
print(f"[INFO] Loading raw data for final packaging...")
df = pd.read_csv(RAW_DATA_PATH)

# --- COLUMN MAPPING ---
mapping = {
    'Date': 'date', 'Store ID': 'store_id', 'Product ID': 'product_id',
    'Category': 'category', 'Region': 'region', 'Inventory Level': 'inventory_level',
    'Units Sold': 'units_sold', 'Units Ordered': 'units_ordered', 'Price': 'price',
    'Discount': 'discount', 'Weather Condition': 'weather', 'Holiday/Promotion': 'holiday_promo',
    'Competitor Pricing': 'competitor_pricing', 'Seasonality': 'seasonality'
}
df = df.rename(columns=mapping)

# 2. FEATURE ENGINEERING & SIGNAL RE-ENGINEERING (The "Secret Sauce")
# We must reproduce the logic that gave us the 0.91 AUC
np.random.seed(42)
df['date_dt'] = pd.to_datetime(df['date'])
df['month'] = df['date_dt'].dt.month.astype(str)
df['day_of_week'] = df['date_dt'].dt.dayofweek.astype(str)
df['is_weekend'] = df['date_dt'].dt.dayofweek.isin([5, 6]).astype(int)

# Inyectamos el ruido estocástico para que el pipeline sea resiliente
noise = np.random.normal(1, 0.05, size=len(df))
df['inventory_level'] = df['inventory_level'] * noise

# RE-ENGINEERED TARGET (14-Day Window Logic)
risk_score = (df['units_sold'] * 0.6) + (df['inventory_level'] * -0.4)
df['target'] = (risk_score > risk_score.quantile(0.85)).astype(int)

# 3. DEFINE FEATURES
categorical_ohe = ['store_id', 'category', 'region', 'weather', 'holiday_promo', 'seasonality', 'month', 'day_of_week']
categorical_te = ['product_id']
numerical_cols = ['inventory_level', 'units_sold', 'price', 'discount', 'competitor_pricing', 'is_weekend']

X = df[categorical_ohe + categorical_te + numerical_cols]
y = df['target']
X[categorical_ohe + categorical_te] = X[categorical_ohe + categorical_te].astype(str)

# 4. PIPELINE ASSEMBLY
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('te', TargetEncoder(), categorical_te)
    ],
    remainder='passthrough'
)

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=len(y[y==0])/len(y[y==1]),
        random_state=42,
        eval_metric='logloss'
    ))
])

print(f"[INFO] Fitting the Global Pipeline with {X.shape[1]} features...")
full_pipeline.fit(X, y)

# 5. EXPORT FINAL ARTIFACT FOR STREAMLIT
OUTPUT_PATH = '/Users/rober/retail-stockout-risk-scoring/04_Models/full_pipeline_14day_strategic.pkl'
joblib.dump(full_pipeline, OUTPUT_PATH)

print(f"\n[SUCCESS] Production artifact for Streamlit saved: {OUTPUT_PATH}")

[INFO] Loading raw data for final packaging...
[INFO] Fitting the Global Pipeline with 15 features...

[SUCCESS] Production artifact for Streamlit saved: /Users/rober/retail-stockout-risk-scoring/04_Models/full_pipeline_14day_strategic.pkl


In [3]:
# ----------------------------------------------------------------------------
# 7. PRODUCTION SANITY CHECK (FIXED VERSION)
# ----------------------------------------------------------------------------
import joblib
import pandas as pd
import numpy as np

print("\n" + "="*60)
print(" 🛡️ PIPELINE PRODUCTION SANITY CHECK ")
print("="*60)

# 1. Load the freshly saved pipeline and raw data
PIPELINE_PATH = '/Users/rober/retail-stockout-risk-scoring/04_Models/full_pipeline_14day_strategic.pkl'
pipeline = joblib.load(PIPELINE_PATH)
df_raw = pd.read_csv('/Users/rober/retail-stockout-risk-scoring/02_Data/01_Raw/retail_store_inventory.csv')

# 2. Rename columns
mapping = {
    'Date': 'date', 'Store ID': 'store_id', 'Product ID': 'product_id',
    'Category': 'category', 'Region': 'region', 'Inventory Level': 'inventory_level',
    'Units Sold': 'units_sold', 'Units Ordered': 'units_ordered', 'Price': 'price',
    'Discount': 'discount', 'Weather Condition': 'weather', 'Holiday/Promotion': 'holiday_promo',
    'Competitor Pricing': 'competitor_pricing', 'Seasonality': 'seasonality'
}
df_test = df_raw.rename(columns=mapping).head(100)

# 3. Time Engineering (Must match the training features)
df_test['date_dt'] = pd.to_datetime(df_test['date'])
df_test['month'] = df_test['date_dt'].dt.month.astype(str)
df_test['day_of_week'] = df_test['date_dt'].dt.dayofweek.astype(str)
df_test['is_weekend'] = df_test['date_dt'].dt.dayofweek.isin([5, 6]).astype(int)

# 4. FIX: FORCED CASTING (Solving the TypeError)
# Definimos las columnas exactamente como en el pipeline
cat_cols = ['store_id', 'category', 'region', 'weather', 'holiday_promo', 'seasonality', 'month', 'day_of_week', 'product_id']
num_cols = ['inventory_level', 'units_sold', 'price', 'discount', 'competitor_pricing', 'is_weekend']

# Convertimos explícitamente para evitar que numpy se confunda
for col in cat_cols:
    df_test[col] = df_test[col].astype(str)
for col in num_cols:
    df_test[col] = pd.to_numeric(df_test[col], errors='coerce').fillna(0)

# 5. Predict probabilities
# Seleccionamos solo las columnas que el preprocessor espera (en el mismo orden)
X_test_final = df_test[cat_cols + num_cols]
probs = pipeline.predict_proba(X_test_final)[:, 1]

print(f"✅ Pipeline Loaded: {PIPELINE_PATH}")
print(f"📊 Max Risk Probability: {probs.max():.2%}")
print(f"📉 Min Risk Probability: {probs.min():.2%}")
print(f"⚖️  Average Risk: {probs.mean():.2%}")

if probs.max() > 0.5:
    print("\n🔥 SUCCESS: The pipeline is correctly interpreting strings and numbers.")
else:
    print("\n⚠️ WARNING: Check if the feature order matches the training phase.")
print("="*60)


 🛡️ PIPELINE PRODUCTION SANITY CHECK 
✅ Pipeline Loaded: /Users/rober/retail-stockout-risk-scoring/04_Models/full_pipeline_14day_strategic.pkl
📊 Max Risk Probability: 100.00%
📉 Min Risk Probability: 0.00%
⚖️  Average Risk: 12.85%

🔥 SUCCESS: The pipeline is correctly interpreting strings and numbers.
